In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables），避免把密钥写进代码
from dotenv import load_dotenv
# 导入标准库 os：读环境变量，例如 OPENAI_API_KEY
import os
# 从 openai 导入 OpenAI 客户端类：既可调云端 Chat Completions，也可指向本地 Ollama 的 OpenAI 兼容端点
from openai import OpenAI
# 从 IPython.display 导入展示工具：Markdown 渲染、display 首次显示、update_display 流式刷新（本格导入备用）
from IPython.display import Markdown, display, update_display



In [ ]:
# ========== 常量：模型名与本地地址集中写在一处 ==========

# OpenAI 云端小模型 id：字符串必须和 API 接受的模型名一致，不要改写
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'
# Ollama 的 OpenAI 兼容 Base URL（/v1）；后面用 OpenAI SDK 指向这里即可调本地模型
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [ ]:
# ========== 环境准备：加载密钥、创建两个客户端、校验 API Key、定义 system prompt ==========

# 加载 .env：override=True 表示用文件里的值覆盖已存在的环境变量
load_dotenv(override=True)
# 默认 OpenAI 客户端：密钥从环境变量 OPENAI_API_KEY 自动读取
openai = OpenAI()
# 第二个客户端指向本地 Ollama；api_key='ollama' 只是占位（本地端点通常不校验真实密钥）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# 默认选用的模型名常量（本练习后续显式传 MODEL_GPT / MODEL_LLAMA，此变量可作对照）
MODEL = MODEL_LLAMA
# 从环境变量取出 OpenAI API Key，便于做形态检查
api_key = os.getenv('OPENAI_API_KEY')

# 粗查密钥：是否存在、是否以 sk-proj- 开头、长度是否够（启发式，不是官方校验）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    # 运行时提示字符串保持英文原样（影响行为的可运行文案不翻译）
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    


# system prompt：发给模型的角色/格式指令；必须保留英文原文，改译会改变回答风格
system_prompt = """
You are a seasoned engineer helping users
understand technical problems by using well defined explanations; when a 
question involves code, include the original code 
in a properly formatted markdown code block and provide
a clear, concise explanation of what the code does and
why it is used, ensuring all responses are structured 
and written in markdown format.
"""


In [ ]:
# ========== 核心函数：流式调用 Chat Completions，边收边刷新 Markdown ==========

def theExplainer(question, model, llm):
    # 向传入的 llm 客户端发起流式聊天：model 决定用哪个模型；messages 含 system + user
    stream = llm.chat.completions.create(
    model=model,
    messages=[{"role": "system", "content": system_prompt}, 
    {"role": "user", "content": question}],
    # stream=True：不要等整段生成完，而是持续返回增量 delta
    stream=True)
    # response：累积已收到的全部文本，用于每次刷新完整 Markdown
    response=""
    # display_id=True：拿到可更新的显示句柄，后面用 .update 原地刷新，而不是不断新打一段
    display_handle = display(Markdown(""), display_id=True)
    # 逐块遍历流式响应
    for chunk in stream:
        # delta.content 可能为 None（例如结束标记）；用 or '' 避免把 None 拼进去
        response += chunk.choices[0].delta.content or ''
        # 用当前完整 response 刷新笔记本里的 Markdown 显示
        display_handle.update(Markdown(response))


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 用户问题（user message）保持英文：这是发给模型的内容，翻译会改变任务语义
# 练习建议：换成你自己看不懂的一行代码，再分别跑下面 GPT / Llama 两格做对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 用云端 gpt-4o-mini 流式解答 ==========

# 传入 OpenAI 云端客户端 openai，以及模型常量 MODEL_GPT
theExplainer(question, MODEL_GPT, openai)


In [ ]:
# ========== 用本地 Llama 3.2（经 Ollama OpenAI 兼容端点）流式解答 ==========

# 传入指向本地的 ollama 客户端，以及模型常量 MODEL_LLAMA；可对比两边回答风格
theExplainer(question, MODEL_LLAMA, ollama)
